# 🚀 BTC 机器学习量化交易策略

## 基于 XGBoost + LSTM 集成学习的比特币量化交易系统

---

### 📋 目录

1. [环境初始化与配置](#1-环境初始化与配置)
2. [数据获取](#2-数据获取)
3. [数据预处理与探索性分析](#3-数据预处理与探索性分析)
4. [特征工程](#4-特征工程)
5. [模型训练与评估 - XGBoost](#5-模型训练与评估---xgboost)
6. [模型训练与评估 - LSTM](#6-模型训练与评估---lstm)
7. [模型集成与策略设计](#7-模型集成与策略设计)
8. [回测执行](#8-回测执行)
9. [回测结果分析与投资建议](#9-回测结果分析与投资建议)
10. [实盘部署指南](#10-实盘部署指南)

---

**策略概述**：使用 XGBoost 和 LSTM 双模型集成预测 BTC/USDT 4小时K线的价格方向，通过加权投票融合信号，结合 Kelly 公式仓位管理和 ATR 动态风控执行交易。

| 参数 | 值 |
|------|-----|
| 交易标的 | BTC/USDT |
| 时间周期 | 4h |
| 数据范围 | 2020-01 至今 |
| 初始资金 | 10,000 USDT |
| 手续费 | 0.1% |
| 滑点 | 0.05% |

## 1. 环境初始化与配置

本节完成以下工作：
- 检查 Python 环境版本
- 安装/验证所有依赖包
- 设置全局配置参数
- 初始化项目路径

In [ ]:
# ============================================
# 1.1 环境检查与依赖安装
# ============================================
import sys
import os

print(f"Python 版本: {sys.version}")
print(f"工作目录: {os.getcwd()}")

# 检查 Python 版本（需要 3.8+）
assert sys.version_info >= (3, 8), "需要 Python 3.8 或更高版本"
print("✅ Python 版本检查通过")

# 安装依赖（如果尚未安装）
# 取消下面的注释来安装依赖
# !pip install -r requirements.txt -q

In [ ]:
# ============================================
# 1.2 导入核心库并验证版本
# ============================================
import warnings
warnings.filterwarnings('ignore')

# 数据处理
import pandas as pd
import numpy as np

# 可视化
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

# 机器学习
import sklearn
import xgboost as xgb

# 深度学习
import tensorflow as tf

# 统计分析
import scipy
import statsmodels

# 交互组件
import ipywidgets as widgets
from tqdm.notebook import tqdm

# 数据获取
import ccxt

# 配置管理
from dotenv import load_dotenv

# 打印版本信息
print("📦 核心库版本信息：")
print(f"  pandas:      {pd.__version__}")
print(f"  numpy:       {np.__version__}")
print(f"  scikit-learn:{sklearn.__version__}")
print(f"  xgboost:     {xgb.__version__}")
print(f"  tensorflow:  {tf.__version__}")
print(f"  plotly:      {go.__version__ if hasattr(go, '__version__') else 'OK'}")
print(f"  ccxt:        {ccxt.__version__}")
print("\n✅ 所有依赖库导入成功")

In [ ]:
# ============================================
# 1.3 全局配置参数
# ============================================

# 加载环境变量
load_dotenv()

# 项目路径配置
PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
DATA_RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')

# 确保目录存在
for dir_path in [DATA_RAW_DIR, DATA_PROCESSED_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(dir_path, exist_ok=True)

# 添加 src 到 Python 路径
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

# ==================== 策略全局配置 ====================
CONFIG = {
    # --- 交易标的与数据 ---
    'symbol': 'BTC/USDT',           # 交易对
    'timeframe': '4h',              # 主时间周期
    'start_date': '2020-01-01',     # 数据起始日期
    'exchange_id': 'binance',       # 交易所
    
    # --- 回测参数 ---
    'initial_capital': 10000,       # 初始资金 (USDT)
    'commission_rate': 0.001,       # 手续费率 0.1%
    'slippage_rate': 0.0005,        # 滑点率 0.05%
    
    # --- 模型参数 ---
    'train_ratio': 0.70,            # 训练集比例
    'val_ratio': 0.15,              # 验证集比例
    'test_ratio': 0.15,             # 测试集比例
    'lstm_window_size': 24,         # LSTM 滑动窗口大小（24个4h = 4天）
    'xgb_n_trials': 50,            # XGBoost Optuna 优化轮数
    
    # --- 策略参数 ---
    'long_threshold': 0.6,          # 做多信号阈值
    'short_threshold': 0.4,         # 做空信号阈值
    'stop_loss_atr_mult': 2.0,      # 止损 ATR 倍数
    'take_profit_atr_mult': 3.0,    # 止盈 ATR 倍数
    'max_holding_periods': 30,      # 最大持仓周期数（30个4h = 5天）
    'trailing_stop_atr_mult': 1.5,  # 移动止损 ATR 倍数
    
    # --- 集成模型权重 ---
    'xgb_weight': 0.5,              # XGBoost 权重
    'lstm_weight': 0.5,             # LSTM 权重
    
    # --- 实盘配置 ---
    'trading_mode': os.getenv('TRADING_MODE', 'backtest'),  # backtest/paper/live
    'api_key': os.getenv('BINANCE_API_KEY', ''),
    'api_secret': os.getenv('BINANCE_SECRET', ''),
}

# 打印配置摘要
print("⚙️ 全局配置参数：")
print(f"  交易标的:   {CONFIG['symbol']}")
print(f"  时间周期:   {CONFIG['timeframe']}")
print(f"  数据起始:   {CONFIG['start_date']}")
print(f"  初始资金:   {CONFIG['initial_capital']:,} USDT")
print(f"  手续费率:   {CONFIG['commission_rate']*100}%")
print(f"  滑点率:     {CONFIG['slippage_rate']*100}%")
print(f"  交易模式:   {CONFIG['trading_mode']}")
print(f"  做多阈值:   {CONFIG['long_threshold']}")
print(f"  做空阈值:   {CONFIG['short_threshold']}")
print("\n✅ 配置加载完成")

## 2. 数据获取

通过 ccxt 库连接 Binance 交易所，获取 BTC/USDT 从 2020年1月至今的 4小时 K线数据。

**数据字段说明**：
- `timestamp`: 时间戳
- `open`: 开盘价
- `high`: 最高价
- `low`: 最低价
- `close`: 收盘价
- `volume`: 成交量

**缓存机制**：首次获取后数据保存为本地 CSV，后续可直接从本地加载。

In [ ]:
# 数据获取模块将在后续 Cell 中实现
# from data_fetcher import DataFetcher

## 3. 数据预处理与探索性分析

对原始 K 线数据进行清洗和探索性分析：
- 缺失值处理（前向填充）
- 异常值检测（Z-score / IQR 方法）
- 收益率分布分析
- 平稳性检验（ADF 检验）
- 自相关性分析（ACF/PACF）

In [ ]:
# 数据预处理模块将在后续 Cell 中实现

## 4. 特征工程

构建机器学习模型所需的多维度特征：

| 特征类别 | 具体指标 |
|---------|----------|
| 技术指标 | MA, EMA, RSI, MACD, 布林带, ATR, ADX, CCI, Williams %R |
| 动量特征 | 多周期收益率, 价格动量, 成交量动量 |
| 波动率特征 | 历史波动率, Parkinson, Garman-Klass |
| 时间特征 | 小时/星期/月份周期编码 |
| 滞后特征 | Lag 1-5, 滚动窗口统计 |

In [ ]:
# 特征工程模块将在后续 Cell 中实现
# from feature_engineer import FeatureEngineer

## 5. 模型训练与评估 - XGBoost

使用 XGBoost 梯度提升树进行价格方向分类预测：
- 使用 Optuna 贝叶斯优化超参数
- Walk-Forward Validation 时间序列交叉验证
- SHAP 值分析特征重要性

In [ ]:
# XGBoost 模型训练将在后续 Cell 中实现
# from models.xgb_model import XGBModel

## 6. 模型训练与评估 - LSTM

使用 LSTM 长短期记忆网络捕捉时序模式：
- 滑动窗口序列构造（窗口大小 = 24 个 4h 周期 = 4天）
- Dropout 正则化防止过拟合
- EarlyStopping + ReduceLROnPlateau 训练策略

In [ ]:
# LSTM 模型训练将在后续 Cell 中实现
# from models.lstm_model import LSTMModel

## 7. 模型集成与策略设计

### 集成方法
XGBoost 和 LSTM 预测结果通过加权投票融合：
- XGBoost 权重: 0.5
- LSTM 权重: 0.5

### 交易规则
- **做多**: 融合概率 > 0.6
- **做空**: 融合概率 < 0.4
- **止损**: 2.0 × ATR
- **止盈**: 3.0 × ATR
- **仓位**: Kelly 公式 + ATR 动态调整

In [ ]:
# 模型集成与策略设计将在后续 Cell 中实现
# from strategy import MLStrategy

## 8. 回测执行

使用自研事件驱动型回测引擎，在 2020年至今的历史数据上模拟策略执行：
- 初始资金: 10,000 USDT
- 手续费: 0.1%
- 滑点: 0.05%
- 逐 4h K线模拟执行

In [ ]:
# 回测引擎将在后续 Cell 中实现
# from backtester import Backtester

## 9. 回测结果分析与投资建议

全面评估策略表现：
- 核心绩效指标（夏普比率、最大回撤、Calmar比率等）
- 月度/年度收益分析
- 与 Buy & Hold 基准对比
- 参数敏感性测试
- 模型衰减分析
- 结构化投资建议报告

In [ ]:
# 结果分析将在后续 Cell 中实现
# from analyzer import PerformanceAnalyzer

## 10. 实盘部署指南

### ⚠️ 重要风险提示

1. **本策略仅供学习研究**，不构成投资建议
2. 加密货币市场波动极大，可能导致本金全部损失
3. 历史回测表现不代表未来收益
4. 建议先使用 Paper Trading 模式充分验证

### 部署步骤

1. 在 Binance 创建 API 密钥（仅开启交易权限，关闭提现权限）
2. 将密钥填入 `.env` 文件
3. 修改 `TRADING_MODE` 为 `paper`（模拟盘）或 `live`（实盘）
4. 运行实盘交易脚本

In [ ]:
# 实盘接口将在后续 Cell 中实现
# from exchange import BinanceExchange